# C3 Step 2 — machine first pass + clip screening (Colab)

Upload **any** audio files (mp3, m4a, mp4, wav, flac, ogg — mixed is fine).
Run the cells. You get back **one zip** containing, for every clip:

| | |
|---|---|
| `processed/<clip>.wav` | converted to 16 kHz mono — the project standard |
| `rttm_draft/<clip>.rttm` | the model's answer. **This is your baseline.** |
| `draft_labels/<clip>.txt` | ready to import into Audacity, flicker removed |

Plus a **screening table** telling you which clips look worth annotating.

---

## Do these once

1. Accept the terms, logged in to Hugging Face:
   https://huggingface.co/pyannote/speaker-diarization-community-1
2. Get a token: https://huggingface.co/settings/tokens (Read is enough)
3. **Key icon** in the left sidebar → **Add new secret** → name it exactly
   `HF_TOKEN` → paste the token into the *value* box → **Notebook access ON**
4. **Runtime → Change runtime type → T4 GPU → Save**

> The **name** goes in the code. The **token** goes behind the key icon. Never swap them.

In [ ]:
# Cell 1 — check you got a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Nothing shown? Runtime > Change runtime type > T4 GPU > Save, then re-run.

In [ ]:
# Cell 2 — install (2-3 min)
!pip install -q pyannote.audio

print('\n' + '=' * 62)
print('NOW DO THIS:   Runtime  >  Restart session')
print('Then SKIP this cell and carry on from Cell 3.')
print('=' * 62)
# Red "dependency resolver" text is a warning, not a failure.
# Only if Cell 3 says google.colab will not import:
#     !pip install -q "pandas==2.2.2"     then restart again.

In [ ]:
# Cell 3 — check the restart worked, then UPLOAD YOUR AUDIO
import pyannote.audio
from google.colab import userdata          # must not error
print('pyannote.audio', pyannote.audio.__version__, '(need 4.x)')
print('colab tools OK')
print()
print('NOW: click the folder icon on the left, then the upload button,')
print('and pick your audio files. Any format. Then run Cell 4.')

In [ ]:
# Cell 4 — find whatever you uploaded and convert it to the project standard
#
# 16 kHz, mono, 16-bit WAV. Handles mp3 / m4a / mp4 / wav / flac / ogg / opus.
import os, glob, re, subprocess

AUDIO_EXT = ('.mp3','.m4a','.mp4','.wav','.flac','.ogg','.opus','.aac','.wma')
os.makedirs('/content/processed', exist_ok=True)

found = [f for f in sorted(glob.glob('/content/*'))
         if f.lower().endswith(AUDIO_EXT)]

if not found:
    raise SystemExit(
        'No audio found in /content.\n'
        'Click the folder icon on the left, then the upload button, pick your\n'
        'files, wait for the upload to finish, then run this cell again.')

def clip_id(path):
    name = os.path.basename(path)
    while '.' in name:
        name = name.rsplit('.', 1)[0]          # strip every extension
    return re.sub(r'[^A-Za-z0-9_.-]', '_', name)   # spaces etc -> underscore

clips, total = [], 0.0
print(f"{'uploaded file':<34}{'-> clip id':<24}{'length':>9}")
print('-' * 68)
for f in found:
    cid = clip_id(f)
    out = f'/content/processed/{cid}.wav'
    subprocess.run(['ffmpeg','-nostdin','-v','error','-y','-i',f,
                    '-ac','1','-ar','16000','-sample_fmt','s16',
                    '-c:a','pcm_s16le', out], check=True)
    dur = float(subprocess.check_output(
        ['ffprobe','-v','error','-show_entries','format=duration',
         '-of','csv=p=0', out]))
    clips.append((cid, out, dur)); total += dur
    print(f'{os.path.basename(f)[:33]:<34}{cid[:23]:<24}{dur:8.1f}s')

print('-' * 68)
print(f'{len(clips)} clips, {total:.1f} s total ({total/60:.2f} min)')

In [ ]:
# Cell 5 — load the model
import os
os.environ['PYANNOTE_METRICS_ENABLED'] = '0'   # privacy premise: no telemetry

import torch
from google.colab import userdata
from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-community-1',
    token=userdata.get('HF_TOKEN'))            # the NAME of the secret

if pipeline is None:
    raise SystemExit('Pipeline empty — you have not accepted the community-1 '
                     'terms on huggingface.co, or the token is wrong.')

pipeline.to(torch.device('cuda'))
print('community-1 loaded on GPU, telemetry off')

# Do NOT switch to precision-2. It runs on pyannoteAI's servers — your audio
# would leave the machine, breaking consent and the project's premise.

In [ ]:
# Cell 6 — run, then write BOTH the .rttm and the Audacity .txt
#
# PROTOCOL v1: no speaker count is supplied. The model must work it out.
import os, time
from itertools import combinations

MIN_TURN, MERGE_GAP = 0.15, 0.30   # flicker threshold / same-speaker join

for d in ('rttm_draft', 'draft_labels'):
    os.makedirs(f'/content/{d}', exist_ok=True)

def merge_close(segs):
    out = []
    for s, e, spk in sorted(segs):
        if out and out[-1][2] == spk and s - out[-1][1] < MERGE_GAP:
            out[-1] = (out[-1][0], max(out[-1][1], e), spk)
        else:
            out.append((s, e, spk))
    return out

report = []
for cid, wav, dur in clips:
    t0 = time.time()
    ann = pipeline(wav).speaker_diarization

    # --- the model's raw answer: THE BASELINE. Never cleaned, never edited.
    with open(f'/content/rttm_draft/{cid}.rttm', 'w') as fh:
        ann.write_rttm(fh)

    raw = sorted((t.start, t.end, spk) for t, _, spk in ann.itertracks(yield_label=True))
    names = {}
    for _, _, spk in raw:
        names.setdefault(spk, f'spk{len(names)+1:02d}')
    raw = [(s, e, names[k]) for s, e, k in raw]

    # how much overlap did the model itself find? (a floor, not the truth)
    ov = sum(max(0.0, min(b, d2) - max(a, c))
             for (a, b, x), (c, d2, y) in combinations(raw, 2) if x != y)
    speech = sum(e - s for s, e, _ in raw)

    # --- tidied copy for Audacity ONLY
    tidy = merge_close(raw)
    dropped = len([1 for s, e, _ in tidy if e - s < MIN_TURN])
    tidy = merge_close([x for x in tidy if x[1] - x[0] >= MIN_TURN])
    with open(f'/content/draft_labels/{cid}.txt', 'w') as fh:
        for s, e, spk in tidy:
            fh.write(f'{s:.3f}\t{e:.3f}\t{spk}\n')

    report.append({'clip': cid, 'dur': dur, 'spk': len(names),
                   'turns': len(tidy), 'speech': speech,
                   'ov_pct': 100 * ov / speech if speech else 0.0,
                   'dropped': dropped, 'rt': time.time() - t0})

# ---------------- screening table ----------------
print('\n' + '=' * 92)
print('SCREENING — which of these are worth annotating for C3?')
print('=' * 92)
print(f"{'clip':<26}{'len':>8}{'spk':>5}{'turns':>7}{'overlap':>9}{'flicker':>9}   verdict")
print('-' * 92)
for r in sorted(report, key=lambda r: (-r['spk'], -r['ov_pct'])):
    if r['spk'] >= 3 and r['ov_pct'] >= 2.0:
        v = 'GOOD — multi-speaker AND overlap'
    elif r['spk'] >= 3:
        v = 'OK — multi-speaker, little overlap'
    elif r['ov_pct'] >= 2.0:
        v = 'MAYBE — few speakers but overlapping'
    else:
        v = 'WEAK — few speakers, no overlap'
    if r['dur'] < 20:
        v += ' (very short)'
    print(f"{r['clip'][:25]:<26}{r['dur']:7.1f}s{r['spk']:>5}{r['turns']:>7}"
          f"{r['ov_pct']:>8.1f}%{r['dropped']:>9}   {v}")

print('-' * 92)
print('READ THIS BEFORE TRUSTING THE TABLE:')
print('  * The model UNDER-COUNTS speakers — it was wrong on 6 of 7 pilot clips.')
print('    "3 speakers" often means 4. Treat the count as a floor.')
print('  * Overlap shown is what the model DETECTED. Real overlap is higher.')
print('  * So this ranks clips. It does not measure them. Listen before deciding.')
print('  * A clip showing 2 speakers and 0% overlap is still probably not worth')
print('    your time — that direction of error is the reliable one.')

In [ ]:
# Cell 7 — download everything as one zip
!cd /content && zip -qr c3_batch.zip processed rttm_draft draft_labels
from google.colab import files
files.download('/content/c3_batch.zip')

# Unzip into  H:\Research Project\C3 Pilot\  and the three folders land in place:
#     processed\     <- the standardised audio
#     rttm_draft\    <- the baseline. Do not edit.
#     draft_labels\  <- open these in Audacity
#
# Then, per clip:
#     File > Open              processed\<clip>.wav
#     File > Import > Labels   draft_labels\<clip>.txt
#     ...fix by ear...
#     File > Export > Export Labels  ->  corrected_labels\<clip>.txt
#                                        ^ corrected_labels, NOT draft_labels